# 05 章 / 第 3 节 · Unsloth LoRA SFT(主推路径)

## 学习目标
- 用 **Unsloth + LoRA** 在 12GB 卡上跑通 Qwen2.5-1.5B-Instruct 的中文 SFT
- 理解 Unsloth 怎么省显存(custom Triton kernel + 4bit base + 优化的内存布局)
- 学会 chat template 应用 + `DataCollatorForCompletionOnlyLM` 做 prompt mask
- 出**微调前后对话对比** + **显存峰值数字**(面试加分项)

## 对应八股
`liguodongiot/llm-action`:**`llm-interview/llm-ft.md`**(LoRA 章节)

## ⚠️ 运行要求
**Linux / WSL2 + CUDA GPU**(Unsloth 官方只支持 Linux)。Windows 原生跑不动,跳到 02 节(TRL 全参)或在 WSL2 里跑。


## 1. 概念背景

为什么 SFT 主推 Unsloth + LoRA 这条路:

- **Unsloth**:用 custom Triton kernel 把 RMSNorm / RoPE / attention / cross-entropy 全 fuse,
  显存比裸 TRL **少 50%+**,吞吐快 **30%+**,12GB 卡跑 1.5B SFT 不再 OOM。
- **LoRA**:只训 ~1% 参数(`r=16` 配置下,Qwen2.5-1.5B 大概 8.4M trainable),
  收敛快、不容易过拟合、效果接近全参 95%(`r=16` 经验)。
- **配合 4bit base**:Unsloth 默认开 4bit NF4,显存又压一档,8GB 卡也能跑。

**为什么 chat template 必须用对**:base 模型从没见过 `<|im_start|>user`,
SFT 时必须用 tokenizer 自带的 chat template 把 prompt 转成正确格式,
否则模型学到一堆乱码。

**为什么要 mask prompt loss**:只让模型在 response 部分算 loss,
prompt 部分用 `-100` 屏蔽。否则模型会学着复读问题,严重浪费样本。


## 2. 环境检查


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from scripts.env_check import check
check(extras=("unsloth", "liger_kernel"))


## 3. 加载模型(Unsloth 一行搞定 4bit + Flash Attention)


In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 2048
MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct"  # Unsloth 预转 4bit 版,首次拉 ~1.2GB
# 也可以用 "Qwen/Qwen2.5-1.5B-Instruct" 让 Unsloth 现场量化,慢但 OK

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype = None,            # None = 自动选 bf16 / fp16
    load_in_4bit = True,     # NF4 量化 base,LoRA adapter 仍是 bf16
)
print(f"模型加载完成 | dtype: {model.dtype}")
print(f"显存占用: {torch.cuda.memory_allocated()/1e9:.2f} GB")


## 4. 加 LoRA adapter

`target_modules` 选 `q/k/v/o + gate/up/down`(覆盖 attention + MLP),`r=16` 是常见起步值。


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,        # Unsloth 推荐 0,实测无副作用
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # Unsloth 自家实现,比 HF 默认省 30%
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

# 看 trainable 参数占比
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable/1e6:.2f} M  |  total: {total/1e6:.2f} M  |  {trainable/total*100:.2f}%")


## 5. 数据集:Alpaca-zh 中文指令微调

用 `shibing624/alpaca-zh`(~50k 样本),应用 Qwen2.5 chat template。


In [ ]:
from datasets import load_dataset

raw = load_dataset("shibing624/alpaca-zh", split="train")
print(f"原始样本: {len(raw)}")
print(f"字段: {raw.column_names}")
print(f"样例:\n{raw[0]}")


In [ ]:
def format_alpaca(example):
    """把 Alpaca 三段(instruction / input / output)转成 Qwen2.5 chat template。"""
    instr = example["instruction"]
    inp   = example.get("input", "")
    out   = example["output"]
    user_msg = instr if not inp else f"{instr}\n\n{inp}"
    messages = [
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": out},
    ]
    # tokenize=False 返回字符串,留给 SFTTrainer 的 dataset_text_field 用
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

# 抽样跑,避免一次性处理 50k
DATA_SUBSET = 5000   # 教学跑,真训练用全集
ds = raw.shuffle(seed=42).select(range(DATA_SUBSET)).map(format_alpaca, remove_columns=raw.column_names)
print(f"格式化后样本: {len(ds)}")
print(f"\n--- 一条 chat template 后的样本 ---\n{ds[0]['text']}")


## 6. SFTTrainer 配置

关键参数:
- `packing=True` —— 短样本拼成定长 chunk,吞吐提升 2-5x
- `learning_rate=2e-4` —— LoRA 比全参的 1e-5 高一档
- `optim="adamw_8bit"` —— 8bit 优化器再省显存
- `gradient_accumulation_steps=4` —— 等效 batch=16


In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

OUTPUT_DIR = "../checkpoints/05_sft_unsloth_lora"

cfg = SFTConfig(
    output_dir = OUTPUT_DIR,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 4,    # 等效 batch = 16
    warmup_steps = 20,
    max_steps = 200,                    # 教学跑;真训用 num_train_epochs 跑 1-3 epoch
    learning_rate = 2e-4,
    logging_steps = 10,
    save_steps = 100,
    save_total_limit = 2,

    optim = "adamw_8bit",               # 配合 bitsandbytes
    weight_decay = 0.01,
    lr_scheduler_type = "cosine",
    seed = 42,

    # SFT 专属
    max_length = MAX_SEQ_LEN,
    packing = True,                     # ⭐ 必开,2-5x 吞吐
    dataset_text_field = "text",

    bf16 = True,                        # bf16 比 fp16 稳
    report_to = "none",                 # 不连 wandb,只看本地 log
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds,
    args = cfg,
)


## 7. 跑训练 + 显存峰值监控


In [ ]:
import time

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
stats = trainer.train()
elapsed = time.time() - t0

peak_gb = torch.cuda.max_memory_allocated() / 1e9
print(f"\n=== 训练完成 ===")
print(f"  总耗时: {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f"  显存峰值: {peak_gb:.2f} GB")
print(f"  末步 loss: {stats.training_loss:.4f}")


## 8. 微调前后对话对比


In [ ]:
FastLanguageModel.for_inference(model)  # Unsloth 加速推理模式

def chat(prompt: str, max_new_tokens: int = 256) -> str:
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    out = model.generate(
        **inputs,
        max_new_tokens = max_new_tokens,
        do_sample = True,
        temperature = 0.7,
        top_p = 0.9,
        repetition_penalty = 1.05,
    )
    response = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response


# 三个 prompt 测试
prompts = [
    "用一句话解释什么是 LoRA。",
    "请帮我写一首五言绝句,主题是「春雨」。",
    "Python 里 list comprehension 和 generator expression 有什么区别?",
]
for p in prompts:
    print(f"--- 用户: {p} ---")
    print(chat(p))
    print()


## 9. 保存:LoRA adapter + 合并后模型(可选)


In [ ]:
# 只保存 adapter(几十 MB)
model.save_pretrained(OUTPUT_DIR + "/lora_adapter")
tokenizer.save_pretrained(OUTPUT_DIR + "/lora_adapter")
print(f"LoRA adapter 已存:{OUTPUT_DIR}/lora_adapter")

# 合并 adapter 到 base,得到独立模型(GB 级,部署用)
# model.save_pretrained_merged(OUTPUT_DIR + "/merged_16bit", tokenizer, save_method="merged_16bit")
# 或直接转 GGUF / GPTQ:
# model.save_pretrained_gguf(OUTPUT_DIR + "/gguf", tokenizer, quantization_method="q4_k_m")


## 10. 可视化 loss 曲线


In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
steps = [l["step"] for l in log_history if "loss" in l]
losses = [l["loss"] for l in log_history if "loss" in l]

plt.figure(figsize=(8, 3))
plt.plot(steps, losses)
plt.xlabel("step"); plt.ylabel("training loss")
plt.title(f"Unsloth LoRA SFT · Qwen2.5-1.5B · {DATA_SUBSET} samples")
plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f"loss: {losses[0]:.3f} -> {losses[-1]:.3f}")


## 11. 踩坑记录

- **`packing=True` 必须开**:不开的话短样本一堆 padding,GPU 利用率掉到 30%。
- **`gradient_checkpointing="unsloth"` 而非 `True`**:Unsloth 的自家实现比 HF 默认省 30% 显存,这是字面意思的字符串参数。
- **LoRA `target_modules` 选齐 7 个**:只选 q/v 是过时建议(2023 LoRA 原论文配置),现代实测 q/k/v/o + gate/up/down 全选效果最好,显存差异微小。
- **chat template 必须用 tokenizer 自带的**:不要自己拼 `<|im_start|>` 字符串,Qwen 不同版本格式有微妙差异,以 `tokenizer.apply_chat_template` 为准。
- **`save_pretrained_merged("merged_4bit", ...)` 是坑**:Unsloth 文档写 `merged_4bit` 是可选,但实测合并后 4bit base 推理质量比纯 LoRA 上 4bit 差,生产部署用 `merged_16bit` 再用 GPTQ/AWQ 单独量化。
- **Unsloth + Liger 不要同时开**:两者都改 attention/MLP kernel,会冲突。Unsloth 已自带 Liger 同等优化,二选一。

## 12. 延伸阅读

- [Unsloth 官方 SFT 文档](https://docs.unsloth.ai/get-started/fine-tuning-guide)
- [TRL SFTTrainer](https://huggingface.co/docs/trl/sft_trainer)
- 论文:`/paper fetch 2106.09685` (LoRA 原论文)
- 论文:`/paper fetch 2305.14314` (QLoRA)
- [[Slipbox/lora-r-alpha-target-modules]]
- [[Slipbox/unsloth-how-it-works]]
